# PDB Fetch Examples

This notebook demonstrates structure metadata and chain sequence retrieval from RCSB PDB using **`run_pdb_fetch`**.

- **Entry metadata** -- Fetch title, experimental method, and resolution
- **FASTA chains** -- Retrieve chain sequences with protein/nucleotide classification

## Imports

In [ ]:
from bio_programming_tools.tools.database_retrieval import (
    run_pdb_fetch,
    PdbFetchInput,
    PdbFetchConfig,
)

## API Reference

**`PdbFetchInput`**

| Field | Type | Default | Description |
|---|---|---|---|
| `pdb_id` | `str` | *(required)* | PDB accession (e.g. `1LBG`) |
| `operation` | `Literal["entry", "fasta"]` | `"entry"` | Fetch entry metadata or FASTA chains |

**`PdbFetchConfig`**

| Field | Type | Default | Description |
|---|---|---|---|
| `request_timeout_seconds` | `int` | `15` | HTTP timeout per request |
| `http_retries` | `int` | `3` | Max HTTP retries |
| `backoff_seconds` | `float` | `1.0` | Retry backoff multiplier |

**`PdbFetchOutput`**

| Field | Type | Description |
|---|---|---|
| `pdb_id` | `str` | PDB accession queried |
| `title` | `Optional[str]` | Structure title (entry op) |
| `method` | `Optional[str]` | Experimental method (entry op) |
| `resolution` | `Optional[float]` | Resolution in angstroms (entry op) |
| `chains` | `List[PdbChain]` | Parsed chain sequences (fasta op) |
| `source_url` | `Optional[str]` | Request URL |

**`PdbChain`**

| Field | Type | Description |
|---|---|---|
| `chain_id` | `Optional[str]` | Chain identifier from header |
| `header` | `str` | Full FASTA header |
| `sequence` | `str` | Chain sequence |
| `is_protein` | `bool` | Whether this chain is protein |

## 1. Fetch Entry Metadata

Retrieve structure title, experimental method, and resolution for the lac repressor structure.

In [ ]:
inputs = PdbFetchInput(pdb_id="1LBG", operation="entry")

output = run_pdb_fetch(inputs, PdbFetchConfig())

print(f"PDB ID: {output.pdb_id}")
print(f"Title: {output.title}")
print(f"Method: {output.method}")
print(f"Resolution: {output.resolution} A")
print(f"Source: {output.source_url}")

## 2. Fetch FASTA Chains

Retrieve all chain sequences from a PDB entry with automatic protein/nucleotide classification.

In [ ]:
inputs = PdbFetchInput(pdb_id="1LBG", operation="fasta")

output = run_pdb_fetch(inputs, PdbFetchConfig())

print(f"PDB ID: {output.pdb_id}")
print(f"Chains: {len(output.chains)}")
print()

for chain in output.chains:
    chain_type = "protein" if chain.is_protein else "nucleotide"
    preview = chain.sequence[:50] + ("..." if len(chain.sequence) > 50 else "")
    print(f"  Chain {chain.chain_id}: {chain_type}, length={len(chain.sequence)}")
    print(f"    {preview}")